# Predicting NBA Game Winners with Logistic Regression
This notebook pulls every game from the 2025-26 NBA regular season, builds features from each team's recent form, and trains models to predict who wins. At the end it exports the results for a Tableau dashboard that ranks the best team.

**Run the cells from top to bottom, in order.**

## 1. Setup
Install the libraries (only needed once), then import everything.

In [ ]:
%pip install nba_api scikit-learn xgboost matplotlib

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 2. Load the data
One row per team per game, so every game shows up twice (once for each team).

In [ ]:
gamefinder = leaguegamefinder.LeagueGameFinder(
    season_nullable='2025-26',
    season_type_nullable='Regular Season',
    league_id_nullable='00'   # NBA only
)
games = gamefinder.get_data_frames()[0]
games.to_csv('games.csv', index=False)
games.head()

In [ ]:
games.columns.tolist()

In [ ]:
games.describe().round(2)

## 3. Clean the data
Turn the date into a real date, and keep only the 30 NBA teams.

In [ ]:
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])

nba_team_ids = [t['id'] for t in teams.get_teams()]
only_nba_games = games[games['TEAM_ID'].isin(nba_team_ids)].copy()

only_nba_games['TEAM_NAME'].nunique()

### Sanity check: every team should have 82 games

In [ ]:
games_per_team = only_nba_games['TEAM_ABBREVIATION'].value_counts()
assert (games_per_team == 82).all(), f"Teams without 82 games:\n{games_per_team[games_per_team != 82]}"
games_per_team.describe()

## 4. Features
### Home game, opponent, and game number

In [ ]:
only_nba_games = only_nba_games.sort_values(['TEAM_ID', 'GAME_DATE']).reset_index(drop=True)

only_nba_games['IS_HOME'] = only_nba_games['MATCHUP'].str.contains('vs.').astype(int)
only_nba_games['OPPONENT'] = only_nba_games['MATCHUP'].str.split(' ').str[-1]
only_nba_games['GAME_NUMBER'] = only_nba_games.groupby('TEAM_ID').cumcount() + 1

only_nba_games[['TEAM_ABBREVIATION', 'GAME_DATE', 'MATCHUP', 'IS_HOME', 'OPPONENT', 'GAME_NUMBER']].head(10)

### Rolling averages over the last 5 games
`shift(1)` makes sure each game only uses games played *before* it, so the model can't peek at the answer.

In [ ]:
window = 5
stats = ['PTS', 'AST', 'STL', 'TOV', 'FG_PCT']

for stat in stats:
    only_nba_games[f'{stat}_ROLL{window}'] = (
        only_nba_games.groupby('TEAM_ID')[stat]
        .transform(lambda s: s.shift(1).rolling(window).mean())
    )

only_nba_games[['TEAM_ABBREVIATION', 'GAME_DATE', 'PTS', 'PTS_ROLL5', 'AST_ROLL5', 'TOV_ROLL5', 'FG_PCT_ROLL5']].head(10)

### Opponent rolling averages
Match each row with the other team's row from the same game.

In [ ]:
only_nba_games = only_nba_games.drop(columns=[c for c in only_nba_games.columns if c.startswith('OPP_')])

roll_cols = [f'{stat}_ROLL{window}' for stat in stats]

opp = only_nba_games[['GAME_ID', 'TEAM_ID'] + roll_cols].rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', **{c: f'OPP_{c}' for c in roll_cols}}
)

only_nba_games = only_nba_games.merge(opp, on='GAME_ID')
only_nba_games = only_nba_games[only_nba_games['TEAM_ID'] != only_nba_games['OPP_TEAM_ID']]
only_nba_games = only_nba_games.sort_values(['TEAM_ID', 'GAME_DATE']).reset_index(drop=True)

len(only_nba_games)

### Target: did the team win? (1 = win, 0 = loss)

In [ ]:
only_nba_games['TARGET_WIN'] = (only_nba_games['WL'] == 'W').astype(int)
only_nba_games[['TEAM_ABBREVIATION', 'GAME_DATE', 'WL', 'TARGET_WIN']].head()

## 5. X and y, split, and scale

In [ ]:
feature_columns = ['IS_HOME'] + roll_cols + [f'OPP_{c}' for c in roll_cols]

x = only_nba_games[feature_columns].dropna()
y = only_nba_games.loc[x.index, 'TARGET_WIN']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, shuffle=True, random_state=42
)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

### Sanity checks

In [ ]:
x.describe()

In [ ]:
x.isnull().sum()

In [ ]:
y.value_counts()

## 6. Logistic regression

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(x_train_scaled, y_train)

y_pred = model.predict(x_test_scaled)
y_prob = model.predict_proba(x_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")

## 7. XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
xgb_model.fit(x_train_scaled, y_train)

y_pred_xgb = xgb_model.predict(x_test_scaled)
y_prob_xgb = xgb_model.predict_proba(x_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")

## Practice 1 - Finetune the model 1
Try lots of settings for each model and keep the one with the best cross-validated ROC AUC.

In [ ]:
def evaluate(model, X, y, name):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    return pd.Series({'accuracy': accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, prob)}, name=name).round(3)

def tune(X_train, y_train):
    lr_grid = GridSearchCV(
        LogisticRegression(max_iter=1000, random_state=42),
        {'C': [0.001, 0.01, 0.1, 1, 10]},
        cv=5, scoring='roc_auc'
    )
    lr_grid.fit(X_train, y_train)

    xgb_grid = GridSearchCV(
        xgb.XGBClassifier(eval_metric='logloss', random_state=42),
        {
            'n_estimators': [100, 300],
            'max_depth': [2, 3, 4],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.8, 1.0],
        },
        cv=5, scoring='roc_auc'
    )
    xgb_grid.fit(X_train, y_train)
    return lr_grid, xgb_grid

lr_grid, xgb_grid = tune(x_train_scaled, y_train)
best_lr = lr_grid.best_estimator_
best_xgb = xgb_grid.best_estimator_

lr_grid.best_params_, xgb_grid.best_params_

In [ ]:
pd.DataFrame([
    evaluate(model, x_test_scaled, y_test, 'Logistic (original)'),
    evaluate(best_lr, x_test_scaled, y_test, 'Logistic (tuned)'),
    evaluate(xgb_model, x_test_scaled, y_test, 'XGBoost (original)'),
    evaluate(best_xgb, x_test_scaled, y_test, 'XGBoost (tuned)'),
])

## Practice 2 - Feature importance analysis
Which features does each model lean on the most? For logistic regression, a positive coefficient pushes toward a win and a negative one toward a loss.

In [ ]:
importance = pd.DataFrame({
    'feature': feature_columns,
    'lr_coef': best_lr.coef_[0],
    'xgb_importance': best_xgb.feature_importances_,
}).sort_values('lr_coef', key=abs, ascending=False).reset_index(drop=True)

importance.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp = importance.sort_values('lr_coef')
axes[0].barh(imp['feature'], imp['lr_coef'], color=['tab:red' if v < 0 else 'tab:green' for v in imp['lr_coef']])
axes[0].set_title('Logistic regression coefficients')
axes[0].axvline(0, color='black', linewidth=0.8)

imp = importance.sort_values('xgb_importance')
axes[1].barh(imp['feature'], imp['xgb_importance'])
axes[1].set_title('XGBoost feature importance')

plt.tight_layout()
plt.show()

## Practice 3 - Test the model on unseen data 1
The models have never seen the 2024-25 season. We rebuild the same features for it, scale with the scaler we already fit, and test.

In [ ]:
def build_features(df, window=5):
    df = df[df['TEAM_ID'].isin(nba_team_ids)].copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df = df.sort_values(['TEAM_ID', 'GAME_DATE']).reset_index(drop=True)

    df['IS_HOME'] = df['MATCHUP'].str.contains('vs.').astype(int)
    df['TARGET_WIN'] = (df['WL'] == 'W').astype(int)

    # rolling averages of the last 5 games (shift(1) = only earlier games)
    for stat in ['PTS', 'AST', 'STL', 'TOV', 'FG_PCT', 'REB', 'FG3_PCT', 'PLUS_MINUS']:
        df[f'{stat}_ROLL{window}'] = (
            df.groupby('TEAM_ID')[stat].transform(lambda s: s.shift(1).rolling(window).mean())
        )

    # new features (used from Practice 4 on)
    df['WIN_PCT_ROLL10'] = df.groupby('TEAM_ID')['TARGET_WIN'].transform(
        lambda s: s.shift(1).rolling(10, min_periods=5).mean()
    )
    df['REST_DAYS'] = df.groupby('TEAM_ID')['GAME_DATE'].diff().dt.days.clip(upper=7)
    df['BACK_TO_BACK'] = (df['REST_DAYS'] == 1).astype(int)

    # copy every team feature over for the opponent
    team_cols = [c for c in df.columns if 'ROLL' in c] + ['REST_DAYS', 'BACK_TO_BACK']
    opp = df[['GAME_ID', 'TEAM_ID'] + team_cols].rename(
        columns={'TEAM_ID': 'OPP_TEAM_ID', **{c: f'OPP_{c}' for c in team_cols}}
    )
    df = df.merge(opp, on='GAME_ID')
    df = df[df['TEAM_ID'] != df['OPP_TEAM_ID']].reset_index(drop=True)
    return df

unseen_raw = leaguegamefinder.LeagueGameFinder(
    season_nullable='2024-25',
    season_type_nullable='Regular Season',
    league_id_nullable='00'
).get_data_frames()[0]

unseen = build_features(unseen_raw)

x_unseen = unseen[feature_columns].dropna()
y_unseen = unseen.loc[x_unseen.index, 'TARGET_WIN']
x_unseen_scaled = scaler.transform(x_unseen)

pd.DataFrame([
    evaluate(best_lr, x_test_scaled, y_test, 'Logistic - test set (2025-26)'),
    evaluate(best_lr, x_unseen_scaled, y_unseen, 'Logistic - unseen (2024-25)'),
    evaluate(best_xgb, x_test_scaled, y_test, 'XGBoost - test set (2025-26)'),
    evaluate(best_xgb, x_unseen_scaled, y_unseen, 'XGBoost - unseen (2024-25)'),
])

## Practice 4 - Add new features
New clues for the model: rebounds, 3-point %, plus/minus, win % over the last 10 games, days of rest, and back-to-backs, for both the team and the opponent.

In [ ]:
season = build_features(games)

new_feature_columns = feature_columns + [
    'REB_ROLL5', 'OPP_REB_ROLL5',
    'FG3_PCT_ROLL5', 'OPP_FG3_PCT_ROLL5',
    'PLUS_MINUS_ROLL5', 'OPP_PLUS_MINUS_ROLL5',
    'WIN_PCT_ROLL10', 'OPP_WIN_PCT_ROLL10',
    'REST_DAYS', 'OPP_REST_DAYS',
    'BACK_TO_BACK', 'OPP_BACK_TO_BACK',
]

x_new = season[new_feature_columns].dropna()
y_new = season.loc[x_new.index, 'TARGET_WIN']

x_new_train, x_new_test, y_new_train, y_new_test = train_test_split(
    x_new, y_new, test_size=0.2, shuffle=True, random_state=42
)

scaler_new = StandardScaler()
x_new_train_scaled = scaler_new.fit_transform(x_new_train)
x_new_test_scaled = scaler_new.transform(x_new_test)

x_new.shape, x_new.isnull().sum().sum(), y_new.value_counts().to_dict()

## Practice 5 - Finetune the model 2
Same tuning as Practice 1, but with the new features.

In [ ]:
lr_grid_new, xgb_grid_new = tune(x_new_train_scaled, y_new_train)
best_lr_new = lr_grid_new.best_estimator_
best_xgb_new = xgb_grid_new.best_estimator_

pd.DataFrame([
    evaluate(best_lr, x_test_scaled, y_test, 'Logistic (old features)'),
    evaluate(best_lr_new, x_new_test_scaled, y_new_test, 'Logistic (new features)'),
    evaluate(best_xgb, x_test_scaled, y_test, 'XGBoost (old features)'),
    evaluate(best_xgb_new, x_new_test_scaled, y_new_test, 'XGBoost (new features)'),
])

In [ ]:
importance_new = pd.DataFrame({
    'feature': new_feature_columns,
    'lr_coef': best_lr_new.coef_[0],
    'xgb_importance': best_xgb_new.feature_importances_,
}).sort_values('lr_coef', key=abs, ascending=False).reset_index(drop=True)

importance_new.round(3)

## Practice 6 - Test the model on unseen data 2
The new-feature models on the 2024-25 season, next to the old ones.

In [ ]:
x_unseen_new = unseen[new_feature_columns].dropna()
y_unseen_new = unseen.loc[x_unseen_new.index, 'TARGET_WIN']
x_unseen_new_scaled = scaler_new.transform(x_unseen_new)

pd.DataFrame([
    evaluate(best_lr, x_unseen_scaled, y_unseen, 'Logistic (old features)'),
    evaluate(best_lr_new, x_unseen_new_scaled, y_unseen_new, 'Logistic (new features)'),
    evaluate(best_xgb, x_unseen_scaled, y_unseen, 'XGBoost (old features)'),
    evaluate(best_xgb_new, x_unseen_new_scaled, y_unseen_new, 'XGBoost (new features)'),
])

## 8. Final results
Every model and version in one table.

In [ ]:
rows = [
    ('Logistic regression', 'original', 'test (2025-26)', model, x_test_scaled, y_test),
    ('Logistic regression', 'tuned', 'test (2025-26)', best_lr, x_test_scaled, y_test),
    ('Logistic regression', 'tuned + new features', 'test (2025-26)', best_lr_new, x_new_test_scaled, y_new_test),
    ('Logistic regression', 'tuned', 'unseen (2024-25)', best_lr, x_unseen_scaled, y_unseen),
    ('Logistic regression', 'tuned + new features', 'unseen (2024-25)', best_lr_new, x_unseen_new_scaled, y_unseen_new),
    ('XGBoost', 'original', 'test (2025-26)', xgb_model, x_test_scaled, y_test),
    ('XGBoost', 'tuned', 'test (2025-26)', best_xgb, x_test_scaled, y_test),
    ('XGBoost', 'tuned + new features', 'test (2025-26)', best_xgb_new, x_new_test_scaled, y_new_test),
    ('XGBoost', 'tuned', 'unseen (2024-25)', best_xgb, x_unseen_scaled, y_unseen),
    ('XGBoost', 'tuned + new features', 'unseen (2024-25)', best_xgb_new, x_unseen_new_scaled, y_unseen_new),
]

results = pd.DataFrame([
    {'model': m, 'version': v, 'data': d, **evaluate(mdl, X, Y, '').to_dict()}
    for m, v, d, mdl, X, Y in rows
])
results

### Takeaways
These sentences fill in your real numbers automatically.

In [ ]:
def acc(model_name, version, data):
    r = results[(results['model'] == model_name) & (results['version'] == version) & (results['data'] == data)]
    return r['accuracy'].iloc[0]

# best model on the season it never saw
unseen_results = results[results['data'] == 'unseen (2024-25)']
best_idx = unseen_results['roc_auc'].idxmax()
best = results.loc[best_idx]
_, _, _, best_model, X_best, y_best = rows[best_idx]

# how good is it when it's confident?
prob = best_model.predict_proba(X_best)[:, 1]
confidence = np.maximum(prob, 1 - prob)
sure = confidence >= 0.65
right_when_sure = ((prob[sure] >= 0.5).astype(int) == y_best.values[sure]).mean() if sure.any() else None

# home court
home_win_rate = season.loc[season['IS_HOME'] == 1, 'TARGET_WIN'].mean()
home_note = "so playing at home is a real advantage" if home_win_rate > 0.52 else "so home court didn't matter much this season"

# top 3 clues
top3 = importance_new.head(3)
clues = ', '.join(
    f"{f} ({'helps' if c > 0 else 'hurts'} winning chances)"
    for f, c in zip(top3['feature'], top3['lr_coef'])
)

test_acc = acc(best['model'], best['version'], 'test (2025-26)')

takeaways = [
    f"The best model was {best['model']} ({best['version']}). On a whole season it had never seen, "
    f"it picked the winner {best['accuracy']:.0%} of the time, compared with 50% for a coin flip.",

    f"Its ROC AUC was {best['roc_auc']:.2f}, where 0.50 means random guessing and 1.00 means perfect.",

    (f"When the model was at least 65% confident ({sure.mean():.0%} of games), it was right {right_when_sure:.0%} of the time."
     if right_when_sure is not None else
     "The model was never more than 65% confident, which shows how close most NBA games are."),

    f"Home teams won {home_win_rate:.0%} of their games in 2025-26, {home_note}.",

    f"The most important clues were: {clues}.",

    f"Tuning changed logistic regression's test accuracy from {acc('Logistic regression', 'original', 'test (2025-26)'):.0%} "
    f"to {acc('Logistic regression', 'tuned', 'test (2025-26)'):.0%}, and XGBoost's from "
    f"{acc('XGBoost', 'original', 'test (2025-26)'):.0%} to {acc('XGBoost', 'tuned', 'test (2025-26)'):.0%}.",

    f"Adding new features (rest days, back-to-backs, recent win %, and more) changed accuracy on the unseen season from "
    f"{acc(best['model'], 'tuned', 'unseen (2024-25)'):.0%} to {acc(best['model'], 'tuned + new features', 'unseen (2024-25)'):.0%}.",

    f"Accuracy went from {test_acc:.0%} on the test set to {best['accuracy']:.0%} on the unseen season. "
    f"Testing on a different season gives a more honest idea of how the model would do in real life.",

    "The model only knows team stats from recent games. It doesn't know about injuries, trades, or star players "
    "resting, which is a big reason no model can predict every game.",
]

for i, t in enumerate(takeaways, 1):
    print(f"{i}. {t}\n")

## 9. Export for Tableau: who is the best team?
Creates a `tableau` folder with four CSV files:
- `team_rankings.csv`: power ranking of all 30 teams (average chance of beating every other team, home and away), plus predicted vs. actual wins
- `matchup_predictions.csv`: win chances for every possible home/away matchup
- `game_predictions.csv`: the model's prediction for every game, each made by a version of the model that never trained on that game
- `feature_importance.csv`: which stats the model relies on

In [ ]:
os.makedirs('tableau', exist_ok=True)

team_info = pd.DataFrame(teams.get_teams())[['id', 'full_name', 'abbreviation']]
team_info.columns = ['TEAM_ID', 'TEAM_NAME', 'TEAM']

# the final model: best logistic regression settings, with scaling built in
final_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=lr_grid_new.best_params_['C'], max_iter=1000)
)

# ---------- 1. Game-by-game predictions ----------
# each game is predicted by a model that did NOT train on that game
games_pred = season.loc[x_new.index, ['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'WL']].copy()
games_pred['WIN_PROB'] = cross_val_predict(
    final_model, x_new, y_new,
    cv=KFold(5, shuffle=True, random_state=42), method='predict_proba'
)[:, 1]
games_pred['PREDICTED'] = np.where(games_pred['WIN_PROB'] >= 0.5, 'W', 'L')
games_pred['CORRECT'] = (games_pred['PREDICTED'] == games_pred['WL']).astype(int)
games_pred = (games_pred
    .merge(team_info, on='TEAM_ID')
    .merge(team_info.rename(columns={'TEAM_ID': 'OPP_TEAM_ID', 'TEAM_NAME': 'OPP_TEAM_NAME', 'TEAM': 'OPP_TEAM'}), on='OPP_TEAM_ID'))

# ---------- 2. Every possible matchup, using each team's latest form ----------
final_model.fit(x_new, y_new)

stats = ['PTS', 'AST', 'STL', 'TOV', 'FG_PCT', 'REB', 'FG3_PCT', 'PLUS_MINUS']
by_date = season.sort_values('GAME_DATE')
form = by_date.groupby('TEAM_ID').tail(5).groupby('TEAM_ID')[stats].mean().add_suffix('_ROLL5')
form['WIN_PCT_ROLL10'] = by_date.groupby('TEAM_ID').tail(10).groupby('TEAM_ID')['TARGET_WIN'].mean()
form['REST_DAYS'] = 2       # assume normal rest
form['BACK_TO_BACK'] = 0

matchup_rows = []
for home, away in itertools.permutations(form.index, 2):
    row = {'HOME_TEAM_ID': home, 'AWAY_TEAM_ID': away, 'IS_HOME': 1}
    row.update(form.loc[home].to_dict())
    row.update({f'OPP_{c}': v for c, v in form.loc[away].items()})
    matchup_rows.append(row)
matchups = pd.DataFrame(matchup_rows)
matchups['HOME_WIN_PROB'] = final_model.predict_proba(matchups[new_feature_columns])[:, 1]
matchups['AWAY_WIN_PROB'] = 1 - matchups['HOME_WIN_PROB']
matchups = (matchups[['HOME_TEAM_ID', 'AWAY_TEAM_ID', 'HOME_WIN_PROB', 'AWAY_WIN_PROB']]
    .merge(team_info.rename(columns={'TEAM_ID': 'HOME_TEAM_ID', 'TEAM_NAME': 'HOME_TEAM_NAME', 'TEAM': 'HOME_TEAM'}), on='HOME_TEAM_ID')
    .merge(team_info.rename(columns={'TEAM_ID': 'AWAY_TEAM_ID', 'TEAM_NAME': 'AWAY_TEAM_NAME', 'TEAM': 'AWAY_TEAM'}), on='AWAY_TEAM_ID'))
matchups['PREDICTED_WINNER'] = np.where(matchups['HOME_WIN_PROB'] >= 0.5, matchups['HOME_TEAM'], matchups['AWAY_TEAM'])

# ---------- 3. Team rankings: who is the best team? ----------
# power score = average chance of beating all 29 other teams, home and away
as_home = matchups.groupby('HOME_TEAM_ID')['HOME_WIN_PROB'].mean()
as_away = matchups.groupby('AWAY_TEAM_ID')['AWAY_WIN_PROB'].mean()
rankings = pd.DataFrame({'POWER_SCORE': (as_home + as_away) / 2})
rankings.index.name = 'TEAM_ID'
rankings = rankings.join(games_pred.groupby('TEAM_ID').agg(
    AVG_WIN_PROB=('WIN_PROB', 'mean'), MODEL_ACCURACY=('CORRECT', 'mean')))
rankings = rankings.join(season.groupby('TEAM_ID').agg(
    ACTUAL_WINS=('TARGET_WIN', 'sum'), GAMES=('TARGET_WIN', 'size')))
rankings['PREDICTED_WINS'] = (rankings['AVG_WIN_PROB'] * rankings['GAMES']).round(1)
rankings['WINS_VS_PREDICTED'] = rankings['ACTUAL_WINS'] - rankings['PREDICTED_WINS']
rankings['POWER_RANK'] = rankings['POWER_SCORE'].rank(ascending=False, method='min').astype(int)
rankings = rankings.reset_index().merge(team_info, on='TEAM_ID').sort_values('POWER_RANK')

# ---------- 4. Feature importance ----------
features = pd.DataFrame({'FEATURE': new_feature_columns, 'COEF': final_model[-1].coef_[0]})
features['DIRECTION'] = np.where(features['COEF'] > 0, 'Helps winning', 'Hurts winning')
features['ABS_COEF'] = features['COEF'].abs()

# ---------- Save for Tableau ----------
games_pred.to_csv('tableau/game_predictions.csv', index=False)
matchups.to_csv('tableau/matchup_predictions.csv', index=False)
rankings.to_csv('tableau/team_rankings.csv', index=False)
features.to_csv('tableau/feature_importance.csv', index=False)

rankings[['POWER_RANK', 'TEAM_NAME', 'POWER_SCORE', 'ACTUAL_WINS', 'PREDICTED_WINS']].head(10)

## 10. Build the dashboard in Tableau
1. Open Tableau Public (free) → Connect → **Text file** → `tableau/team_rankings.csv`. Add the other three files with **Data → New Data Source**.
2. **Who's the best team?** `TEAM_NAME` on Rows, `POWER_SCORE` on Columns, sort descending, `POWER_SCORE` on Color and Label (format as %).
3. **Matchup predictor.** Right-click `HOME_TEAM_NAME` → Create → Parameter ("Home Team"); same for `AWAY_TEAM_NAME` ("Away Team"). Show both parameters. Make a calculated field `[HOME_TEAM_NAME] = [Home Team] AND [AWAY_TEAM_NAME] = [Away Team]`, put it on Filters (True). Measure Values on Columns (only `HOME_WIN_PROB` and `AWAY_WIN_PROB`), Measure Names on Color, `PREDICTED_WINNER` on Label.
4. **Predicted vs. actual wins.** `PREDICTED_WINS` on Columns, `ACTUAL_WINS` on Rows, `TEAM` on Label, `WINS_VS_PREDICTED` on Color (red-green diverging).
5. **What matters most?** `FEATURE` on Rows, `COEF` on Columns, `DIRECTION` on Color, sort by `ABS_COEF`.
6. **Dashboard.** New Dashboard, size Automatic, add a title, drag in the sheets, then File → Save to Tableau Public.